# Titanic Survival Prediction — phiên bản người mới

Đây là lần đầu mình thử tự xây một mô hình phân loại hoàn chỉnh. Mục tiêu là dự đoán một hành khách có sống sót hay không bằng logistic regression tự viết với NumPy.

Mình không cố tối ưu điểm Kaggle. Mình chỉ muốn hiểu từng bước: dữ liệu thiếu ở đâu, tại sao chọn feature, xác suất được tạo ra thế nào và threshold biến xác suất thành nhãn 0/1 ra sao.

## 1. Chuẩn bị môi trường

pandas giúp mình xử lý bảng, Matplotlib giúp vẽ hình. Sigmoid, loss, gradient descent, metric và threshold sẽ được tự viết bằng NumPy.

In [ ]:
import os
import platform
from pathlib import Path

# Đặt cache trong thư mục đang chạy để dùng được cả trong sandbox.
CACHE_DIR = Path.cwd() / ".cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("XDG_CACHE_HOME", str(CACHE_DIR))
os.environ.setdefault("MPLCONFIGDIR", str(CACHE_DIR / "matplotlib"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
np.random.seed(SEED)
np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.max_columns", 100)

print("Python:", platform.python_version())
print("NumPy :", np.__version__)
print("pandas:", pd.__version__)

## 2. Tải dữ liệu

Nếu chạy trong project, cell dưới tự tìm thư mục data. Nếu chạy trên Google Colab và chưa có dữ liệu, hãy upload cùng lúc train.csv, test.csv và gender_submission.csv.

In [ ]:
DATA_CANDIDATES = [
    Path("data"),
    Path("Titanic-Survival-Prediction/data"),
    Path("/content"),
    Path("/content/data"),
]

DATA_DIR = next(
    (path for path in DATA_CANDIDATES
     if (path / "train.csv").exists() and (path / "test.csv").exists()),
    None,
)

if DATA_DIR is None:
    try:
        from google.colab import files
        print("Hãy upload train.csv, test.csv và gender_submission.csv")
        files.upload()
        DATA_DIR = Path.cwd()
    except ImportError as exc:
        raise FileNotFoundError(
            "Không tìm thấy train.csv và test.csv. Hãy đặt chúng trong thư mục data."
        ) from exc

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
sample_path = DATA_DIR / "gender_submission.csv"
sample_submission = pd.read_csv(sample_path) if sample_path.exists() else None

assert "Survived" in train_df.columns
assert "Survived" not in test_df.columns
assert train_df.drop(columns="Survived").columns.equals(test_df.columns)
assert train_df["PassengerId"].is_unique
assert test_df["PassengerId"].is_unique

print("Đọc dữ liệu từ:", DATA_DIR.resolve())
print("Train:", train_df.shape)
print("Test :", test_df.shape)
display(train_df.head())

## 3. Nhìn nhanh target và dữ liệu thiếu

Survived bằng 1 nghĩa là sống sót, bằng 0 nghĩa là không sống sót. Mình chỉ xem số lượng hai nhãn và các ô trống, chưa phân tích tỷ lệ sống theo từng feature.

In [ ]:
target_count = train_df["Survived"].value_counts().sort_index()
target_table = pd.DataFrame({
    "Nhãn": ["0 - Không sống sót", "1 - Sống sót"],
    "Số hành khách": [target_count.get(0, 0), target_count.get(1, 0)],
})
display(target_table)

missing_table = pd.DataFrame({
    "Train thiếu": train_df.isna().sum(),
    "Train thiếu (%)": train_df.isna().mean() * 100,
    "Test thiếu": test_df.isna().sum(),
    "Test thiếu (%)": test_df.isna().mean() * 100,
})
missing_table = missing_table[missing_table.max(axis=1) > 0]
display(missing_table.round(2))

## 4. Chia 80% train và 20% kiểm tra

Mình xáo trộn index bằng seed 42 rồi chia dữ liệu. Mọi mean, mode, danh sách category và chuẩn hóa về sau chỉ được học từ 80% train.

In [ ]:
rng = np.random.default_rng(SEED)
indices = rng.permutation(len(train_df))
split_position = int(0.80 * len(train_df))

train_indices = indices[:split_position]
holdout_indices = indices[split_position:]

train_part = train_df.iloc[train_indices].copy()
holdout_part = train_df.iloc[holdout_indices].copy()
kaggle_test_part = test_df.copy()

y_train = train_part["Survived"].to_numpy(dtype=float)
y_holdout = holdout_part["Survived"].to_numpy(dtype=int)

print("80% để học      :", len(train_part), "hành khách")
print("20% để kiểm tra :", len(holdout_part), "hành khách")
print("Kaggle test.csv :", len(kaggle_test_part), "hành khách")

## 5. Chọn và tạo feature bằng suy luận đơn giản

Mình chọn các feature mà bản thân có thể kể lại lý do:

- Pclass: hạng vé có thể liên quan đến vị trí và điều kiện trên tàu.
- Age: trẻ em và người lớn có thể được ưu tiên khác nhau.
- Fare: giá vé phần nào thể hiện điều kiện chuyến đi.
- Sex: đây là một thông tin trực tiếp, dễ đổi thành 0/1.
- Embarked: cảng lên tàu có thể đại diện cho những nhóm hành khách khác nhau.
- Title: danh xưng lấy từ Name có thể gợi ý tuổi và vai trò trong gia đình.
- FamilySize và IsAlone: đi cùng gia đình có thể khác với đi một mình.
- HasCabin: mình không dùng mã Cabin phức tạp, chỉ hỏi hành khách có thông tin cabin hay không.

PassengerId chỉ là mã định danh. Ticket quá lộn xộn. Name và Cabin nguyên văn được bỏ sau khi đã lấy Title và HasCabin.

In [ ]:
FINAL_FEATURES = [
    "Pclass", "Age", "Fare", "SexBinary",
    "FamilySize", "IsAlone", "HasCabin",
    "Title", "Embarked",
]
CATEGORICAL_FEATURES = ["Title", "Embarked"]

def make_titanic_features(df):
    result = df.copy()

    title = result["Name"].str.extract(r",\s*([^.]*)\.", expand=False)
    title = title.replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    result["Title"] = title.where(
        title.isin(["Mr", "Mrs", "Miss", "Master"]),
        "Rare",
    )

    result["FamilySize"] = result["SibSp"] + result["Parch"] + 1
    result["IsAlone"] = (result["FamilySize"] == 1).astype(int)
    result["HasCabin"] = result["Cabin"].notna().astype(int)
    result["SexBinary"] = result["Sex"].map({"male": 0, "female": 1})

    return result[FINAL_FEATURES]

train_features_raw = make_titanic_features(train_part)
holdout_features_raw = make_titanic_features(holdout_part)
kaggle_features_raw = make_titanic_features(kaggle_test_part)

assert train_features_raw["SexBinary"].notna().all()
assert holdout_features_raw["SexBinary"].notna().all()
assert kaggle_features_raw["SexBinary"].notna().all()

display(train_features_raw.head())

## 6. Điền dữ liệu thiếu

Tuổi trung bình của Master, Miss, Mr và Mrs rõ ràng có thể khác nhau. Vì vậy mình điền Age bằng mean của đúng nhóm Title trong 80% train. Nhóm không có mean sẽ dùng mean tuổi toàn bộ 80% train.

Fare thiếu được điền train mean. Embarked thiếu được điền train mode. Cabin không cần điền vì mình đã đổi nó thành HasCabin.

In [ ]:
# Chỉ học các giá trị điền từ 80% train.
age_mean_by_title = train_features_raw.groupby("Title")["Age"].mean().to_dict()
global_age_mean = float(train_features_raw["Age"].mean())
fare_mean = float(train_features_raw["Fare"].mean())
embarked_mode = str(train_features_raw["Embarked"].mode().iloc[0])

fill_summary = pd.DataFrame({
    "Title": list(age_mean_by_title.keys()),
    "Mean Age dùng để điền": list(age_mean_by_title.values()),
})
display(fill_summary.round(2))
print("Global Age mean:", round(global_age_mean, 2))
print("Fare mean      :", round(fare_mean, 2))
print("Embarked mode  :", embarked_mode)

In [ ]:
def fill_missing_values(df):
    result = df.copy()
    age_from_title = result["Title"].map(age_mean_by_title)
    result["Age"] = result["Age"].fillna(age_from_title).fillna(global_age_mean)
    result["Fare"] = result["Fare"].fillna(fare_mean)
    result["Embarked"] = result["Embarked"].fillna(embarked_mode)
    return result

train_features = fill_missing_values(train_features_raw)
holdout_features = fill_missing_values(holdout_features_raw)
kaggle_features = fill_missing_values(kaggle_features_raw)

assert not train_features.isna().any().any()
assert not holdout_features.isna().any().any()
assert not kaggle_features.isna().any().any()

print("Ô trống còn lại ở 80% train :", train_features.isna().sum().sum())
print("Ô trống còn lại ở 20% test  :", holdout_features.isna().sum().sum())
print("Ô trống còn lại ở test.csv  :", kaggle_features.isna().sum().sum())

## 7. One-hot Title và Embarked

Title và Embarked là chữ nên không thể đưa thẳng vào phép nhân ma trận. pandas.get_dummies đổi mỗi category thành một cột 0/1. Danh sách cột chỉ được lấy từ 80% train.

In [ ]:
X_train_table = pd.get_dummies(
    train_features, columns=CATEGORICAL_FEATURES, dtype=float
)
ENCODED_COLUMNS = X_train_table.columns.tolist()

def encode_like_train(df):
    encoded = pd.get_dummies(
        df, columns=CATEGORICAL_FEATURES, dtype=float
    )
    return encoded.reindex(columns=ENCODED_COLUMNS, fill_value=0.0)

X_holdout_table = encode_like_train(holdout_features)
X_kaggle_table = encode_like_train(kaggle_features)

assert X_train_table.columns.equals(X_holdout_table.columns)
assert X_train_table.columns.equals(X_kaggle_table.columns)

print("Trước one-hot:", len(FINAL_FEATURES), "feature")
print("Sau one-hot  :", X_train_table.shape[1], "cột số")
print(ENCODED_COLUMNS)

## 8. Chuẩn hóa toàn bộ feature

Fare có thể lớn hơn rất nhiều so với các cột 0/1. Mình áp dụng cùng một quy tắc cho toàn bộ ma trận: trừ mean rồi chia std của 80% train. Cột có std bằng 0 được biến thành toàn số 0.

In [ ]:
def fit_standardizer(X_train):
    X_train = np.asarray(X_train, dtype=float)
    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)
    return mean, std

def transform(X, mean, std):
    X = np.asarray(X, dtype=float)
    safe_std = np.where(std == 0, 1.0, std)
    result = (X - mean) / safe_std
    result[:, std == 0] = 0.0
    return result

X_train_raw = X_train_table.to_numpy(dtype=float)
X_holdout_raw = X_holdout_table.to_numpy(dtype=float)
X_kaggle_raw = X_kaggle_table.to_numpy(dtype=float)

feature_mean, feature_std = fit_standardizer(X_train_raw)
X_train = transform(X_train_raw, feature_mean, feature_std)
X_holdout = transform(X_holdout_raw, feature_mean, feature_std)
X_kaggle = transform(X_kaggle_raw, feature_mean, feature_std)

assert X_train.shape[1] == X_holdout.shape[1] == X_kaggle.shape[1]
assert np.isfinite(X_train).all()
assert np.isfinite(X_holdout).all()
assert np.isfinite(X_kaggle).all()

print("X_train shape  :", X_train.shape)
print("X_holdout shape:", X_holdout.shape)
print("X_kaggle shape :", X_kaggle.shape)
print("Mean vài cột đầu:", X_train.mean(axis=0)[:5].round(4))

## 9. Logistic regression hoạt động thế nào?

Đầu tiên mô hình tính điểm số z bằng phép nhân X @ w + b. Sau đó mình dùng ba công thức chính dưới đây.

Sigmoid ép điểm số z thành xác suất từ 0 đến 1:  
$$p = \frac{1}{1 + e^{-z}}$$

Binary cross-entropy phạt mô hình khi xác suất lệch xa nhãn thật:  
$$BCE = -mean(y\log(p) + (1-y)\log(1-p))$$

Gradient cho biết phải dịch weight và bias theo hướng nào để loss giảm:  
$$dw = \frac{X^T(p-y)}{m}, \qquad db = mean(p-y)$$

In [ ]:
def sigmoid(z):
    z = np.asarray(z, dtype=float)
    result = np.empty_like(z, dtype=float)

    positive = z >= 0
    negative = ~positive
    result[positive] = 1.0 / (1.0 + np.exp(-z[positive]))
    exp_z = np.exp(z[negative])
    result[negative] = exp_z / (1.0 + exp_z)
    return result.item() if result.ndim == 0 else result


def binary_cross_entropy(y, probability):
    y = np.asarray(y, dtype=float).reshape(-1)
    probability = np.asarray(probability, dtype=float).reshape(-1)
    safe_probability = np.clip(probability, 1e-12, 1.0 - 1e-12)
    loss = -np.mean(
        y * np.log(safe_probability)
        + (1.0 - y) * np.log1p(-safe_probability)
    )
    return float(loss)

In [ ]:
def classification_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))

    accuracy = (tp + tn) / y_true.size
    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = (
        2.0 * precision * recall / (precision + recall)
        if precision + recall > 0 else 0.0
    )
    return tp, tn, fp, fn, accuracy, precision, recall, f1


def find_best_threshold(y_true, probabilities):
    thresholds = np.arange(5, 96, 5) / 100.0
    f1_scores = []
    for threshold in thresholds:
        prediction = (probabilities >= threshold).astype(int)
        f1_scores.append(classification_metrics(y_true, prediction)[-1])

    # np.argmax lấy vị trí đầu tiên, nên nếu hòa sẽ chọn threshold nhỏ nhất.
    best_index = int(np.argmax(f1_scores))
    return float(thresholds[best_index])

## 10. Tự viết LogisticRegressionGD

Ban đầu toàn bộ weight và bias bằng 0 nên xác suất đều là 0.5. Mỗi epoch dùng toàn bộ 80% train để cập nhật tham số một lần.

In [ ]:
class LogisticRegressionGD:
    def __init__(self, learning_rate=0.1, epochs=1000):
        self.learning_rate = learning_rate
        self.epochs = epochs

    def fit(self, X, y, print_every=100):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1)
        m, n = X.shape

        self.w_ = np.zeros(n, dtype=float)
        self.b_ = 0.0
        self.loss_history_ = []

        initial_probability = sigmoid(X @ self.w_ + self.b_)
        initial_loss = binary_cross_entropy(y, initial_probability)
        self.loss_history_.append(initial_loss)
        print(f"Epoch {0:4d}/{self.epochs} | BCE loss = {initial_loss:.6f}")

        for epoch in range(1, self.epochs + 1):
            probability = sigmoid(X @ self.w_ + self.b_)
            error = probability - y

            dw = X.T @ error / m
            db = np.mean(error)

            self.w_ -= self.learning_rate * dw
            self.b_ -= self.learning_rate * db

            new_probability = sigmoid(X @ self.w_ + self.b_)
            loss = binary_cross_entropy(y, new_probability)
            self.loss_history_.append(loss)

            if epoch % print_every == 0 or epoch == self.epochs:
                print(f"Epoch {epoch:4d}/{self.epochs} | BCE loss = {loss:.6f}")

        self.loss_history_ = np.asarray(self.loss_history_)
        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        return sigmoid(X @ self.w_ + self.b_)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

## 11. Bấm cell này để nhìn mô hình train

Loss càng nhỏ thì xác suất dự đoán nhìn chung càng gần nhãn thật. Mình in một mốc sau mỗi 100 epoch để vẫn nhìn thấy quá trình học mà output không quá dài.

In [ ]:
model = LogisticRegressionGD(learning_rate=0.1, epochs=1000)
model.fit(X_train, y_train, print_every=100)

assert np.isfinite(model.loss_history_).all()
assert model.loss_history_[-1] < model.loss_history_[0]
print("\nTrain xong!")

## 12. Mảng trọng số mô hình học được

Weight dương đẩy điểm số về phía sống sót, weight âm đẩy về phía không sống sót. Vì mọi cột đã được chuẩn hóa, mình chỉ đọc dấu và độ lớn tương đối, không diễn giải theo đơn vị gốc.

In [ ]:
print("Bias:", round(model.b_, 4))
print("Mảng weights đầy đủ:")
print(model.w_)

weight_table = pd.DataFrame({
    "Feature sau one-hot": ENCODED_COLUMNS,
    "Weight": model.w_,
})
weight_table["Độ lớn"] = weight_table["Weight"].abs()
weight_table = weight_table.sort_values("Độ lớn", ascending=False)
display(weight_table.drop(columns="Độ lớn"))

## 13. Tìm threshold tốt nhất cho F1-score

Threshold 0.5 là lựa chọn mặc định, không phải luật bắt buộc. Mình thử 19 threshold từ 0.05 đến 0.95 trên 20% holdout và chọn F1 cao nhất.

Kaggle Titanic chấm accuracy, còn mình chọn theo F1 để luyện đúng bài threshold tuần 3. Hai mục tiêu này không hoàn toàn giống nhau.

In [ ]:
holdout_probability = model.predict_proba(X_holdout)
thresholds = np.arange(5, 96, 5) / 100.0

threshold_rows = []
for threshold in thresholds:
    prediction = (holdout_probability >= threshold).astype(int)
    tp, tn, fp, fn, accuracy, precision, recall, f1 = (
        classification_metrics(y_holdout, prediction)
    )
    threshold_rows.append({
        "Threshold": threshold,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
    })

threshold_table = pd.DataFrame(threshold_rows)
BEST_THRESHOLD = find_best_threshold(y_holdout, holdout_probability)
threshold_table["Tốt nhất"] = np.where(
    np.isclose(threshold_table["Threshold"], BEST_THRESHOLD), "<--", ""
)

display(threshold_table.round(4))
print("Best threshold theo F1:", BEST_THRESHOLD)

## 14. So sánh baseline, threshold 0.5 và threshold tối ưu

Baseline ngây thơ nhất là luôn đoán 0, tức mọi hành khách đều không sống sót. Confusion matrix dùng thứ tự hàng là nhãn thật và cột là nhãn dự đoán: [[TN, FP], [FN, TP]].

In [ ]:
def metric_row(name, y_true, y_pred):
    tp, tn, fp, fn, accuracy, precision, recall, f1 = (
        classification_metrics(y_true, y_pred)
    )
    return {
        "Cách dự đoán": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
    }

baseline_prediction = np.zeros_like(y_holdout)
prediction_05 = model.predict(X_holdout, threshold=0.5)
best_prediction = model.predict(X_holdout, threshold=BEST_THRESHOLD)

metric_table = pd.DataFrame([
    metric_row("Luôn đoán 0", y_holdout, baseline_prediction),
    metric_row("Logistic - threshold 0.50", y_holdout, prediction_05),
    metric_row(
        f"Logistic - threshold {BEST_THRESHOLD:.2f}",
        y_holdout,
        best_prediction,
    ),
])
display(metric_table.round(4))

In [ ]:
tp, tn, fp, fn, *_ = classification_metrics(y_holdout, best_prediction)
confusion = np.array([[tn, fp], [fn, tp]])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(model.loss_history_, color="royalblue")
axes[0].set_title("BCE loss giảm trong lúc train")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("BCE loss")
axes[0].grid(alpha=0.3)

image = axes[1].imshow(confusion, cmap="Blues")
for row in range(2):
    for column in range(2):
        axes[1].text(
            column, row, confusion[row, column],
            ha="center", va="center", fontsize=14,
        )
axes[1].set_xticks([0, 1], labels=["Đoán 0", "Đoán 1"])
axes[1].set_yticks([0, 1], labels=["Thật 0", "Thật 1"])
axes[1].set_title(f"Confusion matrix - threshold {BEST_THRESHOLD:.2f}")
axes[1].set_xlabel("Nhãn dự đoán")
axes[1].set_ylabel("Nhãn thật")
fig.colorbar(image, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## 15. Dự đoán test.csv và tạo submission

Mình dùng nguyên model đã học từ 80% train và threshold tối ưu F1. Mình không train lại bằng 100% dữ liệu.

In [ ]:
test_probability = model.predict_proba(X_kaggle)
test_prediction = (test_probability >= BEST_THRESHOLD).astype(int)

submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"].to_numpy(),
    "Survived": test_prediction,
})

assert len(submission) == len(test_df)
assert submission.columns.tolist() == ["PassengerId", "Survived"]
assert submission["PassengerId"].is_unique
assert set(submission["Survived"].unique()).issubset({0, 1})
if sample_submission is not None:
    assert submission["PassengerId"].equals(sample_submission["PassengerId"])

default_output_dir = DATA_DIR.parent if DATA_DIR.name == "data" else Path.cwd()
OUTPUT_DIR = Path(os.environ.get("TITANIC_OUTPUT_DIR", default_output_dir))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = OUTPUT_DIR / "submission_titanic_beginner.csv"
submission.to_csv(SUBMISSION_PATH, index=False)

print("Đã tạo:", SUBMISSION_PATH.resolve())
print("Số dòng:", len(submission))
print("Số nhãn 0/1:")
display(submission["Survived"].value_counts().sort_index().to_frame("Số dòng"))
display(submission.head())

try:
    from google.colab import files
    files.download(str(SUBMISSION_PATH))
except ImportError:
    print("Không chạy trên Colab nên không tự tải file xuống.")

## 16. Điều mình rút ra

- Classification không trả giá trị liên tục cuối cùng mà trả xác suất rồi mới dùng threshold để lấy nhãn.
- Dữ liệu thiếu có thể được điền bằng một suy luận đơn giản, ví dụ Age mean theo Title.
- Name và Cabin có thể quá phức tạp ở dạng gốc nhưng vẫn dùng được sau khi rút ra Title và HasCabin.
- Loss giảm trên train chưa đủ; metric phải được xem trên 20% chưa dùng để học.
- Threshold 0.5 không phải lúc nào cũng có F1 tốt nhất.
- Đây vẫn là mô hình đầu tiên, feature engineering còn chủ quan và kết quả phụ thuộc vào lần chia dữ liệu.